In [1]:
import os
import time
import requests
import sqlite3
import pandas as pd
import re
from dotenv import load_dotenv
import json

import re
from thefuzz import fuzz

from pathlib import Path



In [2]:
ROOT = Path.cwd().parent
load_dotenv(ROOT / '.env')
STEAM_KEY = os.getenv('STEAM_API_KEY')
TWITCH_CLIENT_ID = os.getenv('TWITCH_CLIENT_ID')
TWITCH_CLIENT_SECRET = os.getenv('TWITCH_CLIENT_SECRET')
DB_PATH = str(ROOT / 'db' / 'gaming_warehouse.db')
_token_cache = {"token": None, "Expira": 0}


In [3]:
print(f"Steam Key: {'OK' if STEAM_KEY else 'FALTA'}")
print(f"DB: {DB_PATH}")

Steam Key: OK
DB: c:\Proyectos\GameLens\db\gaming_warehouse.db


In [4]:
def obtener_token_igdb():
    ahora = time.time()

    if _token_cache["token"] and ahora < _token_cache["Expira"]:
        return _token_cache["token"]

    respuesta = requests.post(
        "https://id.twitch.tv/oauth2/token",
        params={
            "client_id": TWITCH_CLIENT_ID,
            "client_secret": TWITCH_CLIENT_SECRET,
            "grant_type": "client_credentials"
        }
    )
    respuesta.raise_for_status()

    datos = respuesta.json()
    _token_cache["token"] = datos["access_token"]
    _token_cache["Expira"] = ahora + datos.get("expires_in", 3600) - 300

    print("Token de IGDB actualizado exitosamente")
    return _token_cache["token"]

In [25]:
conn = sqlite3.connect(DB_PATH)
try:
    ids_igdb = pd.read_sql_query(
        "SELECT id_igdb FROM CAT_Juego WHERE id_igdb IS NOT NULL AND id_steam IS NULL",
        conn
    )['id_igdb'].tolist()
finally:
    conn.close()

print(f"Juegos sin id_steam: {len(ids_igdb):,}")

Juegos sin id_steam: 2,998


In [6]:
#ids_igdb

In [7]:
from requests.exceptions import ConnectionError, Timeout

LOTE = 30
steam_map = {}
token = obtener_token_igdb()
headers = {'Client-ID': TWITCH_CLIENT_ID, 'Authorization': f'Bearer {token}'}

MAX_REINTENTOS = 3

for i in range(0, len(ids_igdb), LOTE):
    lote = ids_igdb[i:i+LOTE]
    ids_str = ','.join(str(x) for x in lote)
    
    exito = False
    intentos = 0
    
    while not exito and intentos < MAX_REINTENTOS:
        try:
            respuesta = requests.post(
                "https://api.igdb.com/v4/external_games",
                headers=headers,
                data=f'fields game, uid, url; where game = ({ids_str}); limit 500;',
                timeout=10 
            )
            
            if respuesta.status_code == 200:
                for j in respuesta.json():
                    if 'steampowered.com/app/' in j.get('url', ''):
                        steam_map[j['game']] = j['uid']
                exito = True
            else:
                intentos += 1
                print(f"erro {respuesta.status_code} en lote {i}. Intento {intentos}/{MAX_REINTENTOS}.")
                time.sleep(2)
                
        except (ConnectionError, Timeout) as e:
            intentos += 1
            print(f"Conexion caída en lote {i}. Intento {intentos}/{MAX_REINTENTOS}")
            time.sleep(5) 
    
    if not exito:
        print(f"lote {i} falló permanentemente tras {MAX_REINTENTOS} intentos.")

    time.sleep(0.25)
    
    if (i % 300 == 0) or (i + LOTE >= len(ids_igdb)):
        print(f"Procesados {min(i+LOTE, len(ids_igdb)):,} / {len(ids_igdb):,}")

print(f"Steam IDs encontrados: {len(steam_map):,}")

Token de IGDB actualizado exitosamente
Procesados 30 / 10,176
Procesados 330 / 10,176
Procesados 630 / 10,176
Procesados 930 / 10,176
Procesados 1,230 / 10,176
Procesados 1,530 / 10,176
Procesados 1,830 / 10,176
Procesados 2,130 / 10,176
Procesados 2,430 / 10,176
Procesados 2,730 / 10,176
Procesados 3,030 / 10,176
Procesados 3,330 / 10,176
Procesados 3,630 / 10,176
Procesados 3,930 / 10,176
Procesados 4,230 / 10,176
Procesados 4,530 / 10,176
Procesados 4,830 / 10,176
Procesados 5,130 / 10,176
Procesados 5,430 / 10,176
Procesados 5,730 / 10,176
Procesados 6,030 / 10,176
Procesados 6,330 / 10,176
Procesados 6,630 / 10,176
Procesados 6,930 / 10,176
Procesados 7,230 / 10,176
Procesados 7,530 / 10,176
Procesados 7,830 / 10,176
Procesados 8,130 / 10,176
Procesados 8,430 / 10,176
Procesados 8,730 / 10,176
Procesados 9,030 / 10,176
Procesados 9,330 / 10,176
Procesados 9,630 / 10,176
Procesados 9,930 / 10,176
Procesados 10,176 / 10,176
Steam IDs encontrados: 7,182


In [8]:
for id_igdb, uid in steam_map.items():
    if ',' in str(uid):
        print(f"id_igdb={id_igdb} | uid={uid}")

id_igdb=119402 | uid=292030,378649,378648
id_igdb=54612 | uid=377160,435870,435880,435881,480630,480631,490650
id_igdb=25637 | uid=287700,311340,406540,406560,406580,406590,406600,406610,406620,436030,436040,436050,437230,437231
id_igdb=25268 | uid=55230,901805,55380,55381,55382,55385,55386,55387,55388,55389,55390,55391,55392,55395,55396,55398,55399,55400


In [9]:
conn = sqlite3.connect(DB_PATH)
try:
    cursor = conn.cursor()
    actualizaciones = []
    for id_igdb, uid in steam_map.items():
        primer_uid = int(str(uid).split(',')[0].strip())
        actualizaciones.append((primer_uid, id_igdb))
    
    cursor.executemany(
        "UPDATE OR IGNORE CAT_Juego SET id_steam = ? WHERE id_igdb = ?",
        actualizaciones
    )
    conn.commit()
    print(f"Actualizados: {cursor.rowcount:,} juegos")
except Exception as e:
    conn.rollback()
    print(f"Error: {e}")
finally:
    conn.close()

Actualizados: 7,178 juegos


In [29]:
conn = sqlite3.connect(DB_PATH)
try:
    df_faltantes = pd.read_sql_query(
        "SELECT * FROM CAT_Juego WHERE id_igdb IS NOT NULL AND id_steam IS NULL",
        conn
    )
finally:
    conn.close()



In [31]:
df_faltantes['categoria'].value_counts()

categoria
Juego principal            2056
Expansión                   198
Bundle                      178
DLC                         106
Remasterización              92
Port                         79
Juego ampliado               77
Remake                       60
Mod                          34
Expansión independiente      30
Episodio                     27
Actualización                22
Temporada                    14
Pack                         13
Fork                         12
Name: count, dtype: int64

In [33]:
df_faltantes.head(15)

,juego_id,id_igdb,id_steam,titulo,categoria,fecha_lanzamiento,resumen,historia,url_portada,puntuacion_igdb,conteo_votos_igdb,conteo_dlc,conteo_videos,hltb_historia_principal,hltb_historia_extra,hltb_completacionista
0,13,7346,None,The Legend of Zelda: Breath of the Wild,Juego principal,2017-03-03,The Legend of Zelda: Breath of the Wild is the...,Link is awakened in a room by a voice calling ...,https://images.igdb.com/igdb/image/upload/t_co...,92.602534,2809,2,8,None,None,None
1,19,7331,None,Uncharted 4: A Thief's End,Juego principal,2016-05-10,"Several years after his last adventure, retire...",Sin datos,https://images.igdb.com/igdb/image/upload/t_co...,90.528653,2520,0,15,None,None,None
2,21,434,None,Red Dead Redemption,Juego principal,2010-05-18,"A modern-day Western epic, Red Dead Redemption...","In 1911, former outlaw John Marston is forced ...",https://images.igdb.com/igdb/image/upload/t_co...,89.317338,2415,4,4,None,None,None
3,23,11156,None,Horizon Zero Dawn,Juego principal,2017-02-28,Welcome to a vibrant world rich with the beaut...,Sin datos,https://images.igdb.com/igdb/image/upload/t_co...,87.494492,2321,0,9,None,None,None
4,28,19565,None,Marvel's Spider-Man,Juego principal,2018-09-07,"Starring the world’s most iconic Super Hero, S...",Sin datos,https://images.igdb.com/igdb/image/upload/t_co...,87.191812,2018,3,11,None,None,None
5,29,121,None,Minecraft: Java Edition,Juego principal,2011-11-18,Minecraft focuses on allowing the player to ex...,Minecraft: Java Edition (previously known as M...,https://images.igdb.com/igdb/image/upload/t_co...,85.292246,1991,0,1,None,None,None
6,39,26758,None,Super Mario Odyssey,Juego principal,2017-10-27,Explore incredible places far from the Mushroo...,Bowser has kidnapped Princess Peach once again...,https://images.igdb.com/igdb/image/upload/t_co...,89.046821,1777,0,4,None,None,None
7,40,7334,None,Bloodborne,Juego principal,2015-03-24,An action RPG in which the player embodies a H...,Sin datos,https://images.igdb.com/igdb/image/upload/t_co...,91.615566,1765,0,5,None,None,None
8,48,6036,None,The Last of Us Remastered,Remasterización,2014-07-26,The Last of Us Remastered is an updated releas...,Twenty years after a mutated fungus started tu...,https://images.igdb.com/igdb/image/upload/t_co...,93.354589,1661,0,2,None,None,None
9,50,26192,None,The Last of Us Part II,Juego principal,2020-06-19,The Last of Us Part II is an action-adventure ...,Five years after their dangerous journey acros...,https://images.igdb.com/igdb/image/upload/t_co...,90.077643,1644,0,8,None,None,None


In [34]:
#escpeciones porque son nuevos
excepciones = [
    (990080, 136625),  # Hogwarts Legacy
    (4249110, 880),     # Resident Evil 2 (1998)
    (4249120, 966)      # Resident Evil 3 (1999)
]

with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()
    
    cursor.executemany(
        "UPDATE CAT_Juego SET id_steam = ? WHERE id_igdb = ?",
        excepciones
    )
    
    print(f"Agregue {cursor.rowcount} excepciones usando llaves primarias.")

Agregue 3 excepciones usando llaves primarias.


In [13]:
#diccionario_steam

In [35]:
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query("""
        SELECT
            COUNT(*) as total,
            SUM(CASE WHEN id_steam IS NOT NULL THEN 1 ELSE 0 END) as con_steam,
            SUM(CASE WHEN id_steam IS NULL THEN 1 ELSE 0 END) as sin_steam
        FROM CAT_Juego
    """, conn)

display(df)
conn.close()


,total,con_steam,sin_steam
0,10176,7181,2995


# Agregar elr esto de detalles

In [38]:
COLUMNAS_DETALLE = [
    ("steam_price_initial","REAL"),
    ("steam_price_final","REAL"),
    ("steam_discount_percent","INTEGER"),
    ("metacritic_score","INTEGER"),
    ("recommendations_count","INTEGER"),
    ("achievements_count","INTEGER"),
    ("steam_languages","TEXT"),
    ("pc_requirements_json","TEXT"),
]

with sqlite3.connect(DB_PATH) as conn:
    for col, tipo in COLUMNAS_DETALLE:
        try:
            conn.execute(f"ALTER TABLE CAT_Juego ADD COLUMN {col} {tipo}")
        except sqlite3.OperationalError:
            pass  

In [39]:
def extraer_detalle(appid, info):
    precio  = info.get("price_overview", {})
    return (
        precio.get("initial", 0) / 100,
        precio.get("final",0) / 100,
        precio.get("discount_percent", 0),
        info.get("metacritic",{}).get("score"),
        info.get("recommendations",{}).get("total", 0),
        info.get("achievements",{}).get("total", 0),
        info.get("supported_languages"),
        json.dumps(info.get("pc_requirements", {})),
    )

In [47]:
# LOTE_DETALLES = 7000
LOTE_DETALLES = 40
with sqlite3.connect(DB_PATH) as conn:
    juegos = conn.execute("""
        SELECT juego_id, id_steam, titulo
        FROM CAT_Juego
        WHERE id_steam IS NOT NULL AND steam_price_final IS NULL
        LIMIT ?
    """, (LOTE_DETALLES,)).fetchall()

print(f"jeugos pendientes: {len(juegos):,}")

jeugos pendientes: 34


In [48]:
for juego_id, appid, titulo in juegos:
    try:
        r = requests.get(
            f"https://store.steampowered.com/api/appdetails?appids={appid}&l=spanish",
            timeout=10
        )
        if r.status_code == 429:
            print("li,ite alcanzando esperando 60 segundos"); time.sleep(60); continue

        r.raise_for_status()
        blob = r.json().get(str(appid), {})

        if blob.get("success") and "data" in blob:
            vals = extraer_detalle(appid, blob["data"])
            sql  = """UPDATE CAT_Juego SET
                        steam_price_initial=?, steam_price_final=?, steam_discount_percent=?,
                        metacritic_score=?, recommendations_count=?, achievements_count=?,
                        steam_languages=?, pc_requirements_json=?
                      WHERE juego_id=?"""
            with sqlite3.connect(DB_PATH) as conn:
                conn.execute(sql, (*vals, juego_id))
            print(f" listo {titulo}")
        else:
            with sqlite3.connect(DB_PATH) as conn:
                conn.execute("UPDATE CAT_Juego SET steam_price_final=-1 WHERE juego_id=?", (juego_id,))
            print(f" sin datos: {titulo}")

    except Exception as e:
        print(f" execpcion{titulo}: {e}")

    time.sleep(1)

print("Lote finalizado")

 listo Escape from Tarkov
 listo Sifu
 listo Final Fantasy VII Rebirth
 listo Fable Anniversary
 listo Fallout: New Vegas - Old World Blues
 listo Descent
 listo Endless Space 2
 listo Delta Force 2
 listo Dark Void
 listo Fort Solis
 listo N++
 listo Age of Wonders II: The Wizard's Throne
 listo Dishonored: Dunwall City Trials
 listo Constructor
 listo Godus
 listo De Blob 2
 listo A Mortician's Tale
 listo Nancy Drew: Secret of the Scarlet Hand
 listo Painkiller: Redemption
 listo Golden Idol Mysteries: The Spider of Lanka
 listo Painkiller: Recurring Evil
 listo Perception
 listo Kitten Squad
 listo Across the Obelisk
 listo Paper Sorcerer
 listo Love You to Bits
 listo Europa Universalis V
 listo Kynseed
 listo A Bibelot: Tiret sur Will
 listo Liberated
 listo Nexus: The Jupiter Incident
 listo Hauntii
 listo Tokyo Ghoul:re Call to Exist
 listo Rag Doll Kung Fu
Lote finalizado


In [49]:

conn = sqlite3.connect(DB_PATH)

query = """
SELECT 
    COUNT(*) as Total_Juegos_En_BD,
    SUM(CASE WHEN id_steam IS NOT NULL THEN 1 ELSE 0 END) as Total_Con_ID_Steam,
    SUM(CASE WHEN steam_price_final IS NOT NULL AND steam_price_final != -1 THEN 1 ELSE 0 END) as Detalles_Descargados_Exito,
    SUM(CASE WHEN steam_price_final = -1 THEN 1 ELSE 0 END) as Juegos_Ocultos_O_Borrados,
    SUM(CASE WHEN id_steam IS NOT NULL AND steam_price_final IS NULL THEN 1 ELSE 0 END) as Pendientes_Por_Descargar
FROM CAT_Juego;
"""

df_status = pd.read_sql_query(query, conn)
conn.close()

display(df_status)

,Total_Juegos_En_BD,Total_Con_ID_Steam,Detalles_Descargados_Exito,Juegos_Ocultos_O_Borrados,Pendientes_Por_Descargar
0,10176,7181,7103,78,0
